# Squat LLM Results Inspector

Notebook do analizy wyników LLM dla nagrań squat. Wczytuje wyniki z `results`, pobiera wideo, nakłada błędy na klatki, pokazuje surowy output LLM i porównuje predykcje z ground truth z datasetu MLPSD.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path('/Users/kubak/Desktop/MasterDegree')
PROJECT_ROOT = BASE_DIR / 'github' / 'LanguageModule'
sys.path.insert(0, str(PROJECT_ROOT))

from video_llm_evaluation.constants import ERROR_CLASSES
from video_llm_evaluation.notebook_support import (
    MetricsNotAvailable,
    build_summary_dataframe,
    discover_runs,
    gt_summary_for_video,
    intervals_from_payload,
    load_metrics_report,
    load_mlpsd_dataframe,
    load_results_dataframe,
    predicted_errors_from_payload,
    resolve_video_path,
    select_run,
    select_video,
    show_class_card,
    show_report,
    write_review_page,
)

DATASET_PATH = BASE_DIR / 'dataset' / 'datasets' / 'MLPSD' / 'final_dataset' / 'latest' / 'MLPSD_v2.0_feature_extracted_v3.0.0_all_with_holistic_phases.pkl'
PREPROCESSED_VIDEOS_ROOT = BASE_DIR / 'videos' / 'Nagrania' / 'Squat' / 'preprocessed'
RUNS_ROOT = PROJECT_ROOT / 'results' / 'llm_evaluation'

# ---- Z ktorego runu bierzemy WSZYSTKO: nagrania, predykcje, review app, metryki ----
SELECTED_MODEL = 'gemini-3.8-flash'
SELECTED_FPS = 2.0

SELECTED_VIDEO_ID = None
SELECTED_INDEX = 0
MAX_FRAMES_TO_RENDER = 400
FRAME_STEP = 1


## Wybór runu

Wszystko poniżej — lista nagrań, podgląd klatek, review app i metryki — pochodzi
z runu wybranego przez `SELECTED_MODEL` i `SELECTED_FPS` z komórki konfiguracyjnej.

`video_fps_source = inferred` oznacza run sprzed flagi `--video-fps`: pokazane 1.0
to domyślna wartość API, a nie liczba zapisana przez pipeline.

In [ ]:
runs_df = discover_runs(RUNS_ROOT)
display(runs_df.drop(columns=['results_root']))

RESULTS_ROOT = select_run(runs_df, model=SELECTED_MODEL, video_fps=SELECTED_FPS)
print(f'RESULTS_ROOT = {RESULTS_ROOT}')


## Wczytanie wyników `results`

Ta sekcja ładuje pliki wygenerowane przez CLI: `manifest.csv`, `predictions_json` i, jeśli są dostępne, `raw_responses` oraz `labels_npy`.

In [2]:
results_df = load_results_dataframe(RESULTS_ROOT, videos_root=PREPROCESSED_VIDEOS_ROOT)
display(results_df.head(10))

,video_id,prediction_path,relative_dir,video_path_from_results,duration_s,model_name,prompt_version,predictions,global_confidence,notes,video_path,duration_s_manifest,fps,num_frames,ground_truth_path
0,1-0013-1-5-1-2-0-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,formcheck/1-0013-1-5-1-2-0-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,21.966667,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': True...",0.95,Wszystkie powtórzenia są wyraźnie zbyt płytkie...,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,NaN,NaN,NaN,NaN
1,1-0052-50-8-1-2-0-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,formcheck/1-0052-50-8-1-2-0-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,53.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': True...",0.90,We wszystkich powtórzeniach przysiad jest zbyt...,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,NaN,NaN,NaN,NaN
2,1-0057-70-6-1-2-0-1,/Users/kubak/Desktop/MasterDegree/github/Langu...,formcheck/1-0057-70-6-1-2-0-1,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,27.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': Fals...",0.95,W każdym powtórzeniu widoczne jest odrywanie p...,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,NaN,NaN,NaN,NaN
3,1-0071-1-4-1-2-0-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,formcheck/1-0071-1-4-1-2-0-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,31.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': Fals...",0.90,W każdym powtórzeniu występuje zaokrąglenie od...,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,NaN,NaN,NaN,NaN
4,1-0114-1-6-1-3-0-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,formcheck/1-0114-1-6-1-3-0-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,49.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': True...",0.85,"Przysiady są konsekwentnie zbyt płytkie, biodr...",/Users/kubak/Desktop/MasterDegree/videos/Nagra...,NaN,NaN,NaN,NaN
5,1-0125-1-6-1-3-0-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,formcheck/1-0125-1-6-1-3-0-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,24.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': True...",0.85,"Przysiad wykonywany na maszynie Smitha, co wym...",/Users/kubak/Desktop/MasterDegree/videos/Nagra...,NaN,NaN,NaN,NaN
6,1-0003-50-7-1-3-1-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,formtext/1-0003-50-7-1-3-1-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,51.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': Fals...",0.90,W każdym powtórzeniu widoczne jest odrywanie p...,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,NaN,NaN,NaN,NaN
7,1-0-0-5-1-3-5-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,self_recorded/Stanisław_Gwiazda/1-0-0-5-1-3-5-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,13.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': True...",0.92,Wszystkie powtórzenia wykazują ten sam wzorzec...,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,13.0,60.0,780.0,NaN
8,1-0-1-7-0-1-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,self_recorded/Stanisław_Gwiazda/1-0-1-7-0-1-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,33.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': True...",0.90,W nagraniu widoczne są dwa różne wzorce błędów...,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,33.0,30.0,990.0,NaN
9,1-1-1-12-0-3-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,self_recorded/Stanisław_Gwiazda/1-1-1-12-0-3-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,54.000000,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': True...",0.85,None,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,54.0,30.0,1620.0,NaN


## Wczytanie datasetu MLPSD i mapowanie adnotacji

Ta sekcja ładuje `pkl` z MLPSD i buduje mapowanie video -> ground truth errors.

In [3]:
dataset_df = load_mlpsd_dataframe(DATASET_PATH)
display(dataset_df[['video_path', 'gt_errors', 'fps', 'video_time', 'rep_timestamps']].head(10))
if not dataset_df.empty:
    first_labels = dataset_df.iloc[0]['gt_labels_matrix']
    print('gt_labels_matrix example shape =', getattr(first_labels, 'shape', None))
    print('gt_labels_matrix example dtype =', getattr(first_labels, 'dtype', None))

,video_path,gt_errors,fps,video_time,rep_timestamps
0,squats/lifting/0-0084-50-7-0-2-0-0.mp4,[No-knee-outlet],30.0,41.0,"[(0, 123), (124, 255), (256, 367), (368, 492),..."
1,squats/lifting/0-0086-1-5-0-3-0-0.mp4,[Dominant-hip],30.0,17.0,"[(0, 86), (87, 154), (155, 221), (222, 298), (..."
2,squats/lifting/0-0089-60-5-0-3-0-1.mp4,[],30.0,9.0,"[(0, 63), (64, 126), (127, 191), (192, 258)]"
3,squats/lifting/0-0091-120-4-1-3-0-0.mp4,[],29.0,30.97,"[(0, 146), (147, 323), (324, 572), (573, 928)]"
4,squats/lifting/0-0092-95-5-1-3-0-0.mp4,[],30.0,21.0,"[(7, 105), (128, 224), (250, 355), (399, 505),..."
5,squats/lifting/0-0093-60-11-0-3-0-0.mp4,[Squat-depth],30.0,31.0,"[(0, 85), (86, 186), (187, 264), (265, 347), (..."
6,squats/lifting/0-0094-1-5-1-3-0-0.mp4,[],30.0,27.0,"[(0, 96), (142, 253), (285, 395), (496, 611), ..."
7,squats/self_recorded/Jakub_Kępka/0-0085-30-2-1...,[],29.0,8.01,"[(0, 101), (124, 239)]"
8,squats/self_recorded/Jakub_Kępka/0-0086-30-4-1...,[],29.0,11.01,"[(68, 188), (212, 329)]"
9,squats/self_recorded/Jakub_Kępka/0-0087-30-5-1...,[Knee-collapse],29.0,27.03,"[(16, 134), (147, 263), (286, 402), (424, 535)..."


gt_labels_matrix example shape = (10, 1230)
gt_labels_matrix example dtype = float64


## Wybór nagrania do analizy

Wybierz nagranie po `VIDEO_ID` albo po indeksie z tabeli wyników.

In [4]:
selected_result_row, selected_dataset_row = select_video(results_df, dataset_df, selected_video_id=SELECTED_VIDEO_ID, selected_index=SELECTED_INDEX)
selected_video_path = None if selected_result_row is None else resolve_video_path(selected_result_row.get('video_path', selected_result_row.get('video_path_from_results', '')), videos_root=PREPROCESSED_VIDEOS_ROOT)
selected_video_id = None if selected_result_row is None else str(selected_result_row.get('video_id', ''))

print('selected_video_id =', selected_video_id)
print('selected_video_path =', selected_video_path)
if selected_dataset_row is not None:
    display(selected_dataset_row[['video_path', 'gt_errors', 'fps', 'video_time', 'rep_timestamps']])
if selected_result_row is not None:
    display(pd.DataFrame([selected_result_row]))

selected_video_id = 1-0013-1-5-1-2-0-0
selected_video_path = /Users/kubak/Desktop/MasterDegree/videos/Nagrania/Squat/preprocessed/formcheck/1-0013-1-5-1-2-0-0.mp4


video_path                   squats/lifting/0-0084-50-7-0-2-0-0.mp4
gt_errors                                          [No-knee-outlet]
fps                                                            30.0
video_time                                                     41.0
rep_timestamps    [(0, 123), (124, 255), (256, 367), (368, 492),...
Name: 0, dtype: object

,video_id,prediction_path,relative_dir,video_path_from_results,duration_s,model_name,prompt_version,predictions,global_confidence,notes,video_path,duration_s_manifest,fps,num_frames,ground_truth_path
0,1-0013-1-5-1-2-0-0,/Users/kubak/Desktop/MasterDegree/github/Langu...,formcheck/1-0013-1-5-1-2-0-0,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,21.966667,gemini-3.1-pro-preview,v1,"[{'error_type': 'Squat-depth', 'present': True...",0.95,Wszystkie powtórzenia są wyraźnie zbyt płytkie...,/Users/kubak/Desktop/MasterDegree/videos/Nagra...,NaN,NaN,NaN,NaN


## Wczytanie wideo z `PREPROCESSED_VIDEOS_ROOT`

Ta sekcja otwiera plik wideo, czyta wszystkie klatki i mapuje je na czasy w sekundach.

In [5]:
# Loading frames is handled inside build_video_review_widget().

## Nakładanie błędów LLM na klatki wideo

Ta sekcja rysuje aktywne błędy na każdej klatce i tworzy pomocnicze wideo do podglądu w notebooku.

In [15]:
review_page_path = None
if not results_df.empty:
    review_page_path = write_review_page(
        results_df,
        dataset_df,
        videos_root=PREPROCESSED_VIDEOS_ROOT,
        output_dir=RUNS_ROOT / f'{RESULTS_ROOT.name}__review_app',
        max_frames=MAX_FRAMES_TO_RENDER,
        frame_step=FRAME_STEP,
    )
    print('review_page_path =', review_page_path)

review_page_path = /Users/kubak/Desktop/MasterDegree/github/LanguageModule/results/llm_evaluation/squat_review_app_v3/index.html


In [7]:
import socket
import threading
import webbrowser
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path


def _find_free_port(start: int = 8000, end: int = 8100) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        for port in range(start, end):
            try:
                sock.bind(('127.0.0.1', port))
                return port
            except OSError:
                continue
    raise RuntimeError('No free port found for review server')

if review_page_path is not None:
    output_dir = Path(review_page_path).resolve().parent
    port = _find_free_port(8000, 8100)
    handler = lambda *args, directory=str(output_dir): SimpleHTTPRequestHandler(*args, directory=directory)
    server = ThreadingHTTPServer(('127.0.0.1', port), handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    url = f'http://127.0.0.1:{port}/index.html'
    print('Serving review app at', url)
    webbrowser.open_new_tab(url)
    review_http_server = server
else:
    print('No review_page_path available. Run the review page generation cell first.')

Serving review app at http://127.0.0.1:8000/index.html


## Pobranie błędów prawdziwych z datasetu

Ta sekcja odczytuje ground truth z MLPSD i przygotowuje go do porównania z LLM.

In [8]:
gt_info = gt_summary_for_video(dataset_df, selected_video_path) if selected_video_path is not None else {'gt_errors': [], 'gt_labels_matrix': None, 'gt_active_rows': [], 'gt_row': None}
print('gt_active_rows =', gt_info['gt_active_rows'])
if gt_info['gt_labels_matrix'] is not None:
    print('gt_labels_matrix shape =', getattr(gt_info['gt_labels_matrix'], 'shape', None))
    print('gt_labels_matrix dtype =', getattr(gt_info['gt_labels_matrix'], 'dtype', None))
if gt_info['gt_row'] is not None:
    display(pd.DataFrame([gt_info['gt_row']])[['video_path', 'fps', 'video_time', 'rep_timestamps']])

gt_active_rows = []


In [13]:
if not dataset_df.empty:
    first_labels = dataset_df.iloc[0]['gt_labels_matrix']
    print('gt_labels_matrix example shape =', getattr(first_labels, 'shape', None))
    print('gt_labels_matrix example dtype =', getattr(first_labels, 'dtype', None))

gt_labels_matrix example shape = (10, 1230)
gt_labels_matrix example dtype = float64


## Porównanie błędów LLM z błędami ground truth

Ta sekcja pokazuje predykcje i adnotacje obok siebie.

In [14]:
if selected_result_row is not None and selected_video_path is not None:
    prediction_payload = json.loads(Path(selected_result_row['prediction_path']).read_text(encoding='utf-8'))
    pred_errors = predicted_errors_from_payload(prediction_payload)
    gt_errors = list(gt_info.get('gt_errors', []))
    comparison_df = pd.DataFrame(
        {
            'error_type': ERROR_CLASSES,
            'gt_present': [error in gt_errors for error in ERROR_CLASSES],
            'llm_present': [error in pred_errors for error in ERROR_CLASSES],
        }
    )
    comparison_df['match'] = comparison_df['gt_present'] == comparison_df['llm_present']
    display(comparison_df)

    interval_rows = []
    pred_intervals = intervals_from_payload(prediction_payload)
    for error_type in ERROR_CLASSES:
        interval_rows.append(
            {
                'error_type': error_type,
                'gt': 'present' if error_type in gt_errors else 'absent',
                'llm': pred_intervals.get(error_type, []),
            }
        )
    display(pd.DataFrame(interval_rows))

,error_type,gt_present,llm_present,match
0,Squat-depth,False,True,False
1,Back-round,False,False,True
2,Taking-off-foot,False,False,True
3,Knee-collapse,False,False,True
4,Dominant-hip,False,False,True
5,No-knee-outlet,False,False,True


,error_type,gt,llm
0,Squat-depth,absent,"[(0.0, 3.0), (4.5, 8.0), (10.5, 14.0), (16.5, ..."
1,Back-round,absent,[]
2,Taking-off-foot,absent,[]
3,Knee-collapse,absent,[]
4,Dominant-hip,absent,[]
5,No-knee-outlet,absent,[]


## Metryki — wyniki osobno dla każdej klasy błędu

Artefakty zapisane przez `python -m video_llm_evaluation.cli evaluate`, w rozbiciu
na 6 klas błędów: frame-level, video-level i segment-level przy trzech tolerancjach.

Dotyczą runu wybranego na górze notebooka. Notebook niczego tu nie przelicza —
te same liczby raportuje CLI. Jeśli dla runu nie ma jeszcze metryk, dostaniesz
komunikat z gotową komendą.

In [ ]:
# Run jest juz wybrany w komorce konfiguracyjnej (SELECTED_MODEL / SELECTED_FPS).
# Metryki czytamy z artefaktow zapisanych przez `cli.py evaluate` dla tego runu.

report = None
try:
    report = load_metrics_report(RESULTS_ROOT)
except MetricsNotAvailable as exc:
    print(exc)
else:
    print(f'{RESULTS_ROOT.name}: metryki dla {len(report.evaluated_video_ids)} nagran')


In [ ]:
# Pelny raport: zakres runu -> tabela per klasa -> segment-level -> wykresy
# -> karta kazdego bledu -> macierz nagranie x klasa -> odrzucone -> agregaty.
# Wykresy laduja tez jako PNG w metrics/figures/ wybranego runu.
#
# PER_VIDEO = True  -> rozbicie na poszczegolne nagrania
# PER_VIDEO = False -> czyste statystyki per klasa bledu, zsumowane po runie
PER_VIDEO = True

if report is not None:
    show_report(report, tolerance_s=1.0, save_figures=True, per_video=PER_VIDEO)


In [ ]:
# Pojedyncza klasa bledu do przyjrzenia sie z bliska.
# Zmien FOCUS_ERROR na dowolna z ERROR_CLASSES i uruchom komorke ponownie.
FOCUS_ERROR = 'Squat-depth'

if report is not None:
    show_class_card(report, FOCUS_ERROR, per_video=PER_VIDEO)
